In [ ]:
# ============================================================
# KHWEZI MINING — HEALTH, SAFETY & EQUIPMENT MONITORING
# Corrected build. Plain input() loops. No get_* helpers.
# ============================================================
import pandas as pd
from datetime import datetime

# ---------- 1. LOAD ----------
workers_df     = pd.read_csv("workers.csv")
incidents_df   = pd.read_csv("incidents.csv")
equipment_df   = pd.read_csv("equipment.csv")

workers          = workers_df.to_dict("records")
incidents        = incidents_df.to_dict("records")
equipment_records = equipment_df.to_dict("records")

incident_id_counter = max(int(i["IncidentID"].replace("INC", "")) for i in incidents) + 1

alert_log, alert_keys = [], set()
risk_records = []

# ---------- 2. THRESHOLDS ----------
TEMP_WARN, TEMP_CRIT = 80.0, 100.0
VIB_WARN,  VIB_CRIT  = 5.0, 8.0
AVAIL_LIMIT   = 90.0
ALERT_LIMIT   = 5
PPE_MIN       = 80
FATIGUE_MAX   = 8

# ---------- 3. PLAIN-INPUT LOOP HELPERS ----------
def ask_text(prompt):
    while True:
        v = input(prompt).strip()
        if v:
            return v
        print("  ! Input cannot be empty.")

def ask_int(prompt, low, high):
    while True:
        raw = input(f"{prompt} ({low}-{high}): ").strip()
        try:
            v = int(raw)
        except ValueError:
            print("  ! Enter a whole number.")
            continue
        if low <= v <= high:
            return v
        print(f"  ! Must be between {low} and {high}.")

def ask_date(prompt):
    while True:
        raw = input(f"{prompt} (YYYY-MM-DD): ").strip()
        try:
            datetime.strptime(raw, "%Y-%m-%d")
            return raw
        except ValueError:
            print("  ! Invalid date format.")

def ask_time(prompt):
    while True:
        raw = input(f"{prompt} (HH:MM): ").strip()
        try:
            datetime.strptime(raw, "%H:%M")
            return raw
        except ValueError:
            print("  ! Invalid time format.")

def ask_menu(prompt, options):
    """Case-insensitive match against canonical options; returns canonical string."""
    lookup = {o.lower(): o for o in options}
    while True:
        print(f"\n{prompt}")
        for n, o in enumerate(options, 1):
            print(f"  {n}. {o}")
        raw = input("Enter number or name: ").strip()
        if raw.isdigit() and 1 <= int(raw) <= len(options):
            return options[int(raw) - 1]
        if raw.lower() in lookup:
            return lookup[raw.lower()]
        print("  ! Invalid choice.")

# ---------- 4. ACCESS CONTROL ----------
positions = ["Administrator", "Safety Officer", "Mining Engineer",
             "Maintenance Engineer", "Manager"]
functions = ["View Dashboard", "Worker Safety Data", "Safety Incidents",
             "Equipment Data", "Maintenance Data", "Risk Assessment",
             "Generate Reports", "Manage Users"]
permissions = {
    "Administrator":        functions,
    "Safety Officer":       [f for f in functions if f not in ("Equipment Data", "Maintenance Data", "Manage Users")],
    "Mining Engineer":      [f for f in functions if f != "Manage Users"],
    "Maintenance Engineer": [f for f in functions if f not in ("Worker Safety Data", "Safety Incidents", "Risk Assessment", "Manage Users")],
    "Manager":              [f for f in functions if f != "Manage Users"],
}

def login():
    print("\n=== KHWEZI MINING MONITORING SYSTEM ===")
    print("Positions:", ", ".join(positions))
    lookup = {p.lower(): p for p in positions}
    while True:
        raw = input("\nEnter your position (or 'exit'): ").strip()
        if raw.lower() == "exit":
            return None
        if raw.lower() in lookup:
            p = lookup[raw.lower()]
            print(f"\nWelcome, {p}!")
            return p
        print("  ! Access denied — invalid position.")

# ---------- 5. WORKER RISK (single source of truth) ----------
def compute_worker_risk(w):
    """Returns (level, [messages]). Serious warnings force High."""
    msgs, serious = [], False
    if w["PPE_Compliance_pct"] < PPE_MIN:
        msgs.append(f"Low PPE compliance ({w['PPE_Compliance_pct']}%)"); serious = True
    if w["SafetyTrainingStatus"] in ("Due Soon", "Expired"):
        msgs.append(f"Training {w['SafetyTrainingStatus']}")
    if w["FatigueLevel_1to10"] >= FATIGUE_MAX:
        msgs.append(f"High fatigue ({w['FatigueLevel_1to10']}/10)"); serious = True
    if w["NearMisses"] > 0:
        msgs.append(f"{w['NearMisses']} near miss(es)")
    if w["PreviousIncidents"] > 0:
        msgs.append(f"{w['PreviousIncidents']} prior incident(s)")
    level = "High" if serious else ("Medium" if msgs else "Low")
    return level, msgs

# Reconcile stored RiskLevel with computed logic at load time
for w in workers:
    w["RiskLevel"], w["_warn"] = compute_worker_risk(w)

def view_workers():
    if not workers:
        print("No worker records."); return
    print(f"\n{'WorkerID':<10}{'Name':<22}{'Section':<9}{'PPE%':<6}{'Fatigue':<9}{'Risk':<8}")
    print("-" * 70)
    for w in workers:
        print(f"{w['WorkerID']:<10}{w['Name']:<22}{w['SectionID']:<9}"
              f"{w['PPE_Compliance_pct']:<6}{w['FatigueLevel_1to10']:<9}{w['RiskLevel']:<8}")
        for m in w["_warn"]:
            print(f"    - {m}")

def worker_menu():
    while True:
        c = ask_menu("--- Worker Safety Data ---", ["View Workers", "Back"])
        if c == "Back": return
        view_workers()

# ---------- 6. INCIDENTS ----------
SEV_CLASS_CRIT = ("Critical",)

def display_incidents(lst):
    if not lst:
        print("No incidents found."); return
    print(f"\n{'ID':<9}{'Date':<12}{'Shift':<7}{'Type':<22}{'Severity':<9}{'Class':<9}{'Status':<20}")
    print("-" * 90)
    for i in lst:
        print(f"{i['IncidentID']:<9}{i['Date']:<12}{i['Shift']:<7}"
              f"{i['IncidentType']:<22}{i['Severity']:<9}{i['SeverityClass']:<9}"
              f"{i['IncidentStatus']:<20}")

def add_incident():
    global incident_id_counter
    print("\n--- Add Incident ---")
    inc = {
        "IncidentID": f"INC{incident_id_counter:04d}",
        "Date": ask_date("Date"),
        "Time": ask_time("Time"),
        "Shift": ask_menu("Shift", ["Day", "Night"]),
        "Location": ask_text("Location: "),
        "Department": ask_text("Department: "),
        "MineArea": ask_text("Mine area: "),
        "SectionID": ask_text("Section ID: "),
        "EmployeeID": ask_text("Employee ID: "),
        "IncidentType": ask_text("Incident type: "),
        "Severity": ask_menu("Severity", ["Minor", "Moderate", "LTI"]),
        "SeverityClass": ask_menu("Severity class", ["Low", "High", "Critical"]),
        "InjuryStatus": ask_text("Injury status: "),
        "LostTimeInjury": ask_menu("Lost-time injury", ["Yes", "No"]),
        "DaysLost": ask_int("Days lost", 0, 365),
        "Cause": ask_text("Cause: "),
        "CorrectiveAction": ask_text("Corrective action: "),
        "IncidentStatus": ask_menu("Status", ["Open", "Under Investigation", "Closed"]),
        "ReportedToDMRE": ask_menu("Reported to DMRE", ["Yes", "No"]),
    }
    incidents.append(inc)
    incident_id_counter += 1
    print(f"\nIncident {inc['IncidentID']} added.")

def search_incidents():
    term = ask_text("Search term: ").lower()
    display_incidents([i for i in incidents if term in str(i).lower()])

def filter_incidents():
    field = ask_menu("Filter by", ["Severity", "SeverityClass", "Shift",
                                   "Department", "IncidentStatus", "IncidentType"])
    value = ask_text(f"{field} value: ").lower()
    display_incidents([i for i in incidents if str(i.get(field, "")).lower() == value])

def categorize_incidents():
    field = ask_menu("Categorize by", ["IncidentType", "Severity", "SeverityClass",
                                       "Department", "Shift"])
    groups = {}
    for i in incidents:
        groups.setdefault(i.get(field, "Unknown"), []).append(i)
    for cat in sorted(groups, key=lambda k: -len(groups[k])):
        print(f"\n== {cat} — {len(groups[cat])} record(s)")
        display_incidents(groups[cat])

def count_incidents():
    print(f"\nTotal incidents: {len(incidents)}")
    critical = sum(1 for i in incidents if i["SeverityClass"] in SEV_CLASS_CRIT)
    lti      = sum(1 for i in incidents if i["Severity"] == "LTI")
    days     = sum(i["DaysLost"] for i in incidents)
    print(f"  Critical-class : {critical}")
    print(f"  Lost-time (LTI): {lti}")
    print(f"  Total days lost: {days}")

def analyse_trends():
    print("\n--- Incident Trends ---")
    for field in ["SeverityClass", "Severity", "Shift", "Department", "IncidentType"]:
        counts = {}
        for i in incidents:
            counts[i.get(field, "Unknown")] = counts.get(i.get(field, "Unknown"), 0) + 1
        print(f"\nBy {field}:")
        for k, v in sorted(counts.items(), key=lambda x: -x[1]):
            print(f"  {k:<22} {v}")

def incident_menu():
    actions = ["Add Incident", "View All", "Search", "Filter", "Categorize",
               "Count", "Analyse Trends", "Back"]
    while True:
        c = ask_menu("--- Safety Incidents ---", actions)
        if   c == "Add Incident":   add_incident()
        elif c == "View All":       display_incidents(incidents)
        elif c == "Search":         search_incidents()
        elif c == "Filter":         filter_incidents()
        elif c == "Categorize":     categorize_incidents()
        elif c == "Count":          count_incidents()
        elif c == "Analyse Trends": analyse_trends()
        else:                       return

# ---------- 7. EQUIPMENT ----------
def temp_status(t):
    return "NORMAL" if t < TEMP_WARN else ("WARNING" if t <= TEMP_CRIT else "CRITICAL")

def vib_status(v):
    return "NORMAL" if v < VIB_WARN else ("WARNING" if v <= VIB_CRIT else "CRITICAL")

def view_equipment():
    print(f"\n{'ID':<9}{'Type':<17}{'Temp':<10}{'Vib':<10}{'Avail%':<8}{'Maint':<13}{'Alerts':<7}")
    print("-" * 80)
    for e in equipment_records:
        print(f"{e['EquipmentID']:<9}{e['EquipmentType']:<17}"
              f"{e['Temperature_C']:<6}[{temp_status(e['Temperature_C']):<8}] "
              f"{e['Vibration_mms']:<5}[{vib_status(e['Vibration_mms']):<8}] "
              f"{e['Availability_pct']:<8}{e['MaintenanceStatus']:<13}{e['AlertCount']:<7}")

def search_equipment():
    term = ask_text("Search equipment: ").lower()
    hits = [e for e in equipment_records if term in str(e).lower()]
    if not hits:
        print("No matches."); return
    for e in hits:
        print(f"  {e['EquipmentID']} | {e['EquipmentType']} | avail {e['Availability_pct']}%")

def condition_monitoring():
    print("\n--- Equipment Condition ---")
    for e in equipment_records:
        ts, vs = temp_status(e["Temperature_C"]), vib_status(e["Vibration_mms"])
        if ts == "NORMAL" and vs == "NORMAL":
            continue
        print(f"\n{e['EquipmentID']} — {e['EquipmentType']}")
        print(f"  Temp {e['Temperature_C']}C -> {ts}")
        print(f"  Vib  {e['Vibration_mms']} mm/s -> {vs}")
        if ts == "CRITICAL": print("  CRITICAL: temperature above limit — inspect now.")
        if vs == "CRITICAL": print("  CRITICAL: vibration above limit — inspect now.")

def equipment_menu():
    actions = ["View All", "Search", "Condition Monitoring", "Back"]
    while True:
        c = ask_menu("--- Equipment Data ---", actions)
        if   c == "View All":             view_equipment()
        elif c == "Search":               search_equipment()
        elif c == "Condition Monitoring": condition_monitoring()
        else:                             return

# ---------- 8. MAINTENANCE ----------
def maintenance_flags(e):
    flags = []
    status = str(e.get("MaintenanceStatus", "")).strip().lower()
    if status == "overdue":
        flags.append("OVERDUE")
    elif status == "due":
        flags.append("DUE")
    try:
        due = datetime.strptime(e["NextServiceDue"], "%Y-%m-%d").date()
        days = (due - datetime.today().date()).days
        if days < 0 and "OVERDUE" not in flags:
            flags.append("OVERDUE (date passed)")
    except (ValueError, KeyError):
        pass
    if e["Availability_pct"] < AVAIL_LIMIT:
        flags.append(f"LOW AVAIL {e['Availability_pct']}%")
    if e["AlertCount"] > ALERT_LIMIT:
        flags.append(f"HIGH ALERTS {e['AlertCount']}")
    return flags

def show_maintenance_flags():
    print(f"\n{'ID':<9}{'Type':<17}{'Status':<13}{'Flags'}")
    print("-" * 80)
    for e in equipment_records:
        f = maintenance_flags(e)
        print(f"{e['EquipmentID']:<9}{e['EquipmentType']:<17}{e['MaintenanceStatus']:<13}"
              f"{', '.join(f) if f else 'OK'}")

def maintenance_menu():
    while True:
        c = ask_menu("--- Maintenance Data ---", ["View Status", "Back"])
        if c == "Back": return
        show_maintenance_flags()

# ---------- 9. RISK ASSESSMENT ----------
def classify_risk(score):
    if score <= 4:  return "Low"
    if score <= 9:  return "Medium"
    if score <= 16: return "High"
    return "Critical"

def risk_assessment():
    print("\n--- Risk Assessment (Likelihood x Consequence) ---")
    L = ask_int("Likelihood", 1, 5)
    C = ask_int("Consequence", 1, 5)
    score = L * C
    cls = classify_risk(score)
    risk_records.append({"Likelihood": L, "Consequence": C, "Score": score, "Classification": cls})
    print(f"\nRisk score {score} -> {cls}")
    if cls in ("High", "Critical"):
        print("WARNING: notify Safety Officer and Manager immediately.")

# ---------- 10. ALERTS ----------
def raise_alert(kind, source, level, detail=""):
    key = (kind, str(source), level)
    if key in alert_keys:
        return
    alert_keys.add(key)
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    alert_log.append({"Time": now, "Type": kind, "Source": source,
                      "Level": level, "Detail": detail})
    tag = "CRITICAL" if level == "CRITICAL" else "WARNING"
    print(f"\n[{tag}] {now}  source={source}  issue={kind}" + (f"  ({detail})" if detail else ""))
    print("  Action: inspect and follow procedure.\n")

def check_conditions():
    for e in equipment_records:
        if temp_status(e["Temperature_C"]) == "CRITICAL":
            raise_alert("High Equipment Temperature", e["EquipmentID"], "CRITICAL",
                        f"{e['Temperature_C']}C")
        if vib_status(e["Vibration_mms"]) == "CRITICAL":
            raise_alert("High Equipment Vibration", e["EquipmentID"], "CRITICAL",
                        f"{e['Vibration_mms']} mm/s")
        if "OVERDUE" in " ".join(maintenance_flags(e)):
            raise_alert("Overdue Maintenance", e["EquipmentID"], "WARNING")
    for w in workers:
        if w["FatigueLevel_1to10"] >= FATIGUE_MAX:
            raise_alert("High Worker Fatigue", w["WorkerID"], "WARNING",
                        f"level {w['FatigueLevel_1to10']}")
        if w["PPE_Compliance_pct"] < PPE_MIN:
            raise_alert("Low PPE Compliance", w["WorkerID"], "WARNING",
                        f"{w['PPE_Compliance_pct']}%")
    for i in incidents:
        if i["SeverityClass"] in SEV_CLASS_CRIT:          # FIXED: was i["Severity"]
            raise_alert("Critical Severity Incident", i["IncidentID"], "CRITICAL",
                        i["IncidentType"])

def view_alert_log():
    print("\n--- Alert Log ---")
    if not alert_log:
        print("No alerts recorded."); return
    for a in alert_log:
        print(f"{a['Time']} | {a['Level']:<8} | {a['Source']:<10} | {a['Type']}"
              + (f" | {a['Detail']}" if a["Detail"] else ""))

# ---------- 11. DASHBOARD & REPORTS ----------
def dashboard():
    n_w   = len(workers)
    avg_ppe = round(sum(w["PPE_Compliance_pct"] for w in workers) / n_w, 1) if n_w else 0.0
    high_risk = sum(1 for w in workers if w["RiskLevel"] == "High")
    near_miss = sum(w["NearMisses"] for w in workers)
    n_e   = len(equipment_records)
    avg_av = round(sum(e["Availability_pct"] for e in equipment_records) / n_e, 1) if n_e else 0.0
    under_maint = sum(1 for e in equipment_records if e["MaintenanceStatus"].lower() == "overdue")
    crit_alerts = sum(1 for a in alert_log if a["Level"] == "CRITICAL")

    print("\n" + "=" * 58)
    print("       KHWEZI MINING — MONITORING DASHBOARD")
    print("=" * 58)
    print(f"  Workers ................... {n_w}")
    print(f"  Avg PPE compliance ........ {avg_ppe}%")
    print(f"  High-risk workers ......... {high_risk}")
    print(f"  Near misses (sum) ......... {near_miss}")
    print(f"  Incidents ................. {len(incidents)}")
    print(f"  Equipment ................. {n_e}")
    print(f"  Avg availability .......... {avg_av}%")
    print(f"  Overdue maintenance ....... {under_maint}")
    print(f"  Alerts raised ............. {len(alert_log)}")
    print(f"  Critical alerts ........... {crit_alerts}")
    print("=" * 58)

def generate_report():
    dashboard()
    print("\n--- Equipment Needing Attention ---")
    any_flag = False
    for e in equipment_records:
        f = maintenance_flags(e)
        if f:
            any_flag = True
            print(f"  {e['EquipmentID']:<9} {e['EquipmentType']:<17} {', '.join(f)}")
    if not any_flag:
        print("  (none)")
    print("\n--- Recent Alerts ---")
    view_alert_log()

def manage_users():
    print("\n--- Role Permissions ---")
    for p in positions:
        print(f"\n{p}:")
        for f in permissions[p]:
            print(f"   - {f}")

# ---------- 12. MAIN ----------
handlers = {
    "View Dashboard":     dashboard,
    "Worker Safety Data": worker_menu,
    "Safety Incidents":   incident_menu,
    "Equipment Data":     equipment_menu,
    "Maintenance Data":   maintenance_menu,
    "Risk Assessment":    risk_assessment,
    "Generate Reports":   generate_report,
    "Manage Users":       manage_users,
}

def run_session(position):
    check_conditions()
    while True:
        allowed = permissions[position]
        print(f"\n=== Main Menu — {position} ===")
        for n, f in enumerate(allowed, 1):
            print(f"  {n}. {f}")
        print(f"  {len(allowed)+1}. View Alert Log")
        print(f"  {len(allowed)+2}. Log Out")
        raw = input("Select: ").strip()

        if raw.isdigit():
            k = int(raw)
            if 1 <= k <= len(allowed):
                handlers[allowed[k-1]](); continue
            if k == len(allowed) + 1:
                view_alert_log(); continue
            if k == len(allowed) + 2:
                print("Logging out.\n"); return
        if raw.lower() == "log out":
            print("Logging out.\n"); return
        print("  ! Invalid or restricted option.")

def main():
    print(f"Loaded: {len(workers)} workers | {len(incidents)} incidents | "
          f"{len(equipment_records)} equipment records")
    while True:
        pos = login()
        if pos is None:
            print("Goodbye."); break
        run_session(pos)

main()   

Loaded: 30 workers | 40 incidents | 19 equipment records

=== KHWEZI MINING MONITORING SYSTEM ===
Positions: Administrator, Safety Officer, Mining Engineer, Maintenance Engineer, Manager
